In [ ]:
import torch
import torch.nn as nn
import math

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
src_vocab = {"hello":0, "world":1, "good":2, "morning":3}
tgt_vocab = {"bonjour":0, "monde":1, "bon":2, "matin":3}

src_vocab_size = len(src_vocab)
tgt_vocab_size = len(tgt_vocab)

In [ ]:
src_data = torch.tensor([[0,1],[2,3]]).to(device)
tgt_data = torch.tensor([[0,1],[2,3]]).to(device)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:,0::2] = torch.sin(position * div_term)
        pe[:,1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self,x):
        return x + self.pe[:,:x.size(1)]

In [ ]:
class TransformerModel(nn.Module):

    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=64, nhead=4):
        super().__init__()

        self.src_embed = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embed = nn.Embedding(tgt_vocab_size, d_model)

        self.positional = PositionalEncoding(d_model)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=2,
            num_decoder_layers=2
        )

        self.fc_out = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt):

        src = self.src_embed(src)
        tgt = self.tgt_embed(tgt)

        src = self.positional(src)
        tgt = self.positional(tgt)

        src = src.permute(1,0,2)
        tgt = tgt.permute(1,0,2)

        output = self.transformer(src, tgt)

        output = self.fc_out(output)

        return output

In [ ]:
model = TransformerModel(src_vocab_size, tgt_vocab_size).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(


In [ ]:
epochs = 200

for epoch in range(epochs):

    optimizer.zero_grad()

    output = model(src_data, tgt_data)

    output = output.reshape(-1, tgt_vocab_size)
    target = tgt_data.permute(1,0).reshape(-1)

    loss = criterion(output, target)

    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print("Epoch:", epoch, "Loss:", loss.item())

Epoch: 0 Loss: 1.4684464931488037
Epoch: 50 Loss: 0.012367263436317444
Epoch: 100 Loss: 0.007009544409811497
Epoch: 150 Loss: 0.004301150795072317


In [ ]:
with torch.no_grad():

    output = model(src_data, tgt_data)

    pred_tokens = torch.argmax(output, dim=2)

    pred_tokens = pred_tokens.permute(1,0)

    print("\nPredicted token IDs:")
    print(pred_tokens)

print("\nGenerated Translations:")

for sentence in pred_tokens:

    words = [tgt_idx_to_word[token.item()] for token in sentence]

    print(" ".join(words))


Predicted token IDs:
tensor([[0, 1],
        [2, 3]])

Generated Translations:
bonjour monde
bon matin
